> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 06 · AgentCore Memory: remembering students

**CampusMate v3 (agent that remembers)**  ·  ⏱️ Time budget: **15 min theory + 30 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

A good mentor remembers you're a 3rd-semester student who loves cloud computing without you repeating it every time. Your Lab 5 agent forgets everything when a session ends. **AgentCore Memory** stores conversations and automatically pulls out lasting facts, so CampusMate can still know your interests next week.

### What you'll do
- Learn short-term vs long-term memory, and the two memory strategies we use
- Add a Memory to your project with one CLI command
- Add a few lines to your agent so it uses the memory
- Tell CampusMate a fact, start a NEW session, and confirm it remembers

## Settings

✏️ Change `STUDENT_ID` to your own ID before running anything else.

In [ ]:
# Settings for this lab
import sys, json, time, uuid, pathlib, subprocess, shutil
import boto3

# ✏️ Put YOUR student ID here (the same one in every notebook). Lower-case letters and digits, e.g. priya07
STUDENT_ID = "student01"

REGION = "us-east-1"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"         # Amazon Nova 2 Lite
CHEAP_MODEL_ID = "us.amazon.nova-2-lite-v1:0"
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"

PROJECT = f"campusmate{STUDENT_ID}"                    # your AgentCore project (created in notebook 05)
PROJECT_DIR = ROOT / PROJECT                           # the folder it lives in
AGENT_FILE = PROJECT_DIR / "app" / "agent" / "main.py" # your agent's code

def agentcore(*args, folder=None):
    """Run an AgentCore CLI command (the same one you would type in a terminal) and show its output."""
    process = subprocess.Popen([shutil.which("agentcore") or "agentcore", *args], cwd=folder or PROJECT_DIR,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        if "[SDK info]" not in line:
            print(line, end="")
    if process.wait() != 0:
        raise RuntimeError("The agentcore command failed. Read the messages above.")

print("Student:", STUDENT_ID, "| Project folder:", PROJECT_DIR)

## What you built in earlier labs

The AgentCore CLI knows everything in your project. We ask it, so there are no IDs to copy.

In [ ]:
def deployed():
    """Ask the AgentCore CLI what your project has in AWS: {name: details}."""
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "status", "--json"], cwd=PROJECT_DIR, capture_output=True, text=True)
    return {r["name"]: r for r in json.loads(out.stdout)["resources"]}

def ask(prompt, session_id, student="AIDD-DEMO-001"):
    """Send one message to your deployed agent (agentcore invoke) and return its answer."""
    payload = json.dumps({"prompt": prompt, "student_id": student})
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "invoke", "--json", "--runtime", "agent",
                          "--session-id", session_id, "--payload", payload], cwd=PROJECT_DIR, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stdout + out.stderr)
    return json.loads(json.loads(out.stdout)["body"])["result"]

resources = deployed()
AGENT_ARN = resources["agent"]["arn"]
RUNTIME_ID = AGENT_ARN.split("/")[-1]
LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT"      # your agent's logs and traces
for name, r in resources.items():
    print(f"{r['resourceType']:14s} {name:16s} {r.get('arn', '')}")

## Two kinds of memory

| | Short-term memory | Long-term memory |
|---|---|---|
| What | Every message (an **event**) in a session | **Records** pulled out of the events |
| When | Saved instantly | Pulled out **in the background**, after about 30-90 seconds |
| Scope | One session | All sessions of one student (the **actor**) |
| Campus analogy | Today's attendance register | Your file in the department office |

## Two strategies decide what goes into long-term memory

- **SEMANTIC**: facts and preferences, e.g. *"Student is in 3rd semester; interested in cloud."* Stored under `/users/<student>/facts`.
- **SUMMARIZATION**: a short recap of each session. Stored under `/summaries/<student>/<session>`.

Our **actor ID is the student ID** (`AIDD-DEMO-001`), so each student's memories stay separate.

## Step 1: add a Memory to your project

(In a terminal: `agentcore add memory --name studentMemory --strategies SEMANTIC,SUMMARIZATION`.) This only updates your project file; nothing is created in AWS until you deploy.

In [ ]:
if "studentMemory" in (PROJECT_DIR / "agentcore" / "agentcore.json").read_text():
    print("studentMemory is already in your project")
else:
    agentcore("add", "memory", "--name", "studentMemory", "--strategies", "SEMANTIC,SUMMARIZATION")

## Step 2: update your agent

Same file as Lab 5, with the lines marked **`# NEW in Lab 6`**:
- `MEMORY_ID = os.getenv("MEMORY_STUDENTMEMORY_ID")`: after you deploy, AgentCore gives your agent this setting automatically.
- `memory_for(...)`: saves each message and, before each answer, brings back what is known about this student.
- `session_manager=memory_for(session, student)` plugs it into the agent.

In [ ]:
%%writefile {AGENT_FILE}
import json
from pathlib import Path
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

DATA_DIR = Path(__file__).parent / "data"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"   # Amazon Nova 2 Lite


# ---- Your tools from notebook 04 (unchanged) ----
@tool
def get_course_info(query: str) -> str:
    """Look up AIDD courses by course code (e.g. AIDD303) or a keyword in the title/description (e.g. docker).

    Args:
        query: A course code or a keyword.
    """
    q = query.lower()
    courses = json.loads((DATA_DIR / "courses.json").read_text())["courses"]
    hits = [c for c in courses if q == c["code"].lower() or q in (c["title"] + " " + c["description"]).lower()]
    return json.dumps(hits[:5]) if hits else json.dumps({"found": False, "codes": [c["code"] for c in courses]})


@tool
def get_campus_policy(topic: str) -> str:
    """Return CampusMate rules for a topic: attendance, internal_assessments, semester_end_exams, fees, leave,
    re_evaluation, library, hackathons. Always use this instead of guessing a rule.

    Args:
        topic: one of the topics above
    """
    policies = json.loads((DATA_DIR / "campus_policies.json").read_text())["policies"]
    return json.dumps(policies.get(topic.lower().strip(), {"found": False, "topics": list(policies)}))


# ---- NEW in Lab 6: memory ----
import os
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

MEMORY_ID = os.getenv("MEMORY_STUDENTMEMORY_ID")   # AgentCore fills this in for you after `agentcore add memory`


def memory_for(session, student):
    """Save this conversation, and bring back what we already know about this student."""
    return AgentCoreMemorySessionManager(
        AgentCoreMemoryConfig(memory_id=MEMORY_ID, session_id=session, actor_id=student,
                              retrieval_config={"/users/{actorId}/facts": RetrievalConfig(top_k=5),
                                                "/summaries/{actorId}": RetrievalConfig(top_k=3)}),
        region_name="us-east-1")


# ---- AgentCore Runtime wrapper (Lab 5) ----
app = BedrockAgentCoreApp()
agents = {}   # one agent per conversation (session)


@app.entrypoint
def invoke(payload, context):
    student = payload.get("student_id", "AIDD-DEMO-001")
    session = context.session_id
    if session not in agents:
        agents[session] = Agent(
            model=BedrockModel(model_id=MODEL_ID, max_tokens=1200),
            tools=[get_course_info, get_campus_policy],
            session_manager=memory_for(session, student),   # NEW in Lab 6
            system_prompt=f"You are CampusMate for AIDD students, talking to student {student}. "
                          "Use tools for courses, rules and attendance; never invent rules. Be brief.",
        )
    answer = agents[session](payload["prompt"])
    return {"result": str(answer)}


if __name__ == "__main__":
    app.run()

## Step 3: deploy

This creates the Memory and updates your agent (2-4 minutes; the Memory takes a little while the first time).

In [ ]:
agentcore("deploy")

In [ ]:
def deployed():
    """Ask the AgentCore CLI what your project has in AWS: {name: details}."""
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "status", "--json"], cwd=PROJECT_DIR, capture_output=True, text=True)
    return {r["name"]: r for r in json.loads(out.stdout)["resources"]}

def ask(prompt, session_id, student="AIDD-DEMO-001"):
    """Send one message to your deployed agent (agentcore invoke) and return its answer."""
    payload = json.dumps({"prompt": prompt, "student_id": student})
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "invoke", "--json", "--runtime", "agent",
                          "--session-id", session_id, "--payload", payload], cwd=PROJECT_DIR, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stdout + out.stderr)
    return json.loads(json.loads(out.stdout)["body"])["result"]

resources = deployed()
MEMORY_ID = resources["studentMemory"]["arn"].split("/")[-1]
print("Your memory:", MEMORY_ID)

## Step 4: tell CampusMate something about yourself

We pretend to be student **AIDD-DEMO-001**.

In [ ]:
STUDENT = "AIDD-DEMO-001"
session_a = f"lab06-A-{uuid.uuid4()}"
print(ask("Hi! I'm in 3rd semester and I'm really interested in cloud computing.", session_a, STUDENT))

## Step 5: wait for long-term memory (it works in the background)

Pulling facts out of a conversation happens **in the background**, usually within 30-90 seconds. If we ask a new session too early,
it won't know anything yet. So we check every 15 seconds until the fact appears. This waiting is normal, not a bug.

In [ ]:
memory_api = boto3.client("bedrock-agentcore", region_name=REGION)

def facts_about(student):
    r = memory_api.retrieve_memory_records(memoryId=MEMORY_ID, namespacePath=f"/users/{student}/facts",
                                           searchCriteria={"searchQuery": "semester and interests", "topK": 5})
    return [x["content"]["text"] for x in r.get("memoryRecordSummaries", [])]

for attempt in range(12):            # up to about 3 minutes
    found = facts_about(STUDENT)
    if any("cloud" in f.lower() for f in found):
        break
    print(f"  attempt {attempt + 1}: not there yet, waiting 15 s...")
    time.sleep(15)
print("Long-term facts:", found)

## 🔬 Live check: a brand-new session remembers

In [ ]:
session_b = f"lab06-B-{uuid.uuid4()}"     # NEW session, same student
print(ask("Suggest one topic I should explore next. Use what you know about me.", session_b, STUDENT))

## ✍️ Your turn

Teach CampusMate a **preference** as a different student (for example *'Please always answer me in bullet points'*), wait for it to appear in memory, then check that a new session follows it.

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
OTHER_STUDENT = "AIDD-DEMO-008"
MY_PREFERENCE = "TODO"     # TODO: e.g. "Please always answer me in bullet points."

if MY_PREFERENCE != "TODO":
    print(ask(MY_PREFERENCE, f"pref-{uuid.uuid4()}", OTHER_STUDENT))
    # TODO: wait using facts_about(OTHER_STUDENT) like Step 5, then ask a question in a new session
else:
    print("Skipped: set MY_PREFERENCE first.")

## ✅ Checkpoint

You are on track if:
- Step 5 printed a fact like 'in 3rd semester' / 'interested in cloud computing'
- The new session's suggestion mentions cloud

**Stuck or behind?** Copy from `the Step 2 cell (the whole agent file); finished version: projects/campusmate_agent.py` and keep going. That's what it's there for.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| Facts list stays empty | Long-term memory works in the background and can take a few minutes | Wait and re-run the Step 5 cell |
| `KeyError: 'studentMemory'` | The deploy didn't finish | Re-run Step 3 and read its messages |
| The new session doesn't remember | The fact wasn't extracted yet, or a different student ID was used | Wait for Step 5, use the same `STUDENT` |